> Part of **Complete Python Study Material** — split across per-chapter notebooks. See [`00_index.ipynb`](00_index.ipynb) for the notebook conventions, per-concept template, status tags, the digitalization log, chapter coverage tracker and cross-reference index.

## 7. Exception Handling

*Scope:* How Python signals, propagates and recovers from errors.

### 7.1 Errors versus Exceptions

"Error" is the umbrella term; Python code can go wrong in three different ways, only one
of which this chapter's `try`/`except` machinery can do anything about:

| Category | Detected | Crashes the program? | Catchable with `try`/`except`? |
|---|---|---|---|
| **Syntax error** | *before* execution starts, while parsing | yes, immediately | no — the code never even starts running |
| **Logical error** | never automatically — code runs, gives the *wrong* answer | no | n/a — no exception is ever raised |
| **Exception** | *during* execution, when an operation can't complete | only if nothing catches it | yes — this is what the rest of chapter 7 covers |

A syntax error means the parser rejects the code outright — a missing `:` is invalid
Python, not a runtime event:

In [ ]:
# compile() parses without running -> demonstrates the SyntaxError without a broken cell
try:
    compile("if True\n    pass\n", "<test>", "exec")   # missing ':' after True
except SyntaxError as e:
    print("SyntaxError:", e)   # SyntaxError: expected ':' (<test>, line 1)

A logical error is the opposite extreme — the code is perfectly valid and runs to
completion, it just computes the wrong thing. Nothing raises, nothing crashes, so
`try`/`except` has nothing to catch — the only way to find this bug is to notice the
output is wrong:

In [ ]:
def average(nums):
    return sum(nums) / len(nums) - 1   # bug: that trailing "- 1" shouldn't be there

print(average([2, 4, 6]))   # 3.0 -> wrong (should be 4.0), but runs fine, no exception at all

An exception sits in between: the code is syntactically valid, and *most* of the time it
runs fine, but a specific input at runtime makes an operation impossible to complete —
and unlike a logical error, Python notices and raises an object describing exactly what
went wrong. This is the only category `try`/`except` can act on, and it's the subject of
the rest of this chapter:

In [ ]:
def average(nums):
    return sum(nums) / len(nums)   # correct formula, but len(nums) can be 0

try:
    print(average([]))
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)   # ZeroDivisionError: division by zero -> caught and recoverable

### 7.2 Handling Syntax and Semantics

`try`/`except` lets a program catch a runtime error and recover from it instead of
crashing outright. If nothing catches an exception, it keeps propagating upward — out of
the function that raised it, out of whatever called that function, and so on — until
either something catches it or it reaches the top and stops the program (7.3 covers this
propagation in depth).

In [ ]:
try:
    x = 10 / 0
except ZeroDivisionError as e:
    print("Caught:", e)   # Caught: division by zero -> the crash was intercepted

**Multiple `except` blocks** — a single `try` can be followed by several `except`
blocks, each targeting a different exception type. Python checks them **top to bottom**
and runs only the **first** one that matches:

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Cannot divide by zero")
    except TypeError:
        print("Both arguments must be numbers")

safe_divide(10, 0)     # Cannot divide by zero
safe_divide(10, "a")     # Both arguments must be numbers
print(safe_divide(10, 2))   # 5.0 -> no exception at all, the return value comes straight through

**Common mistake** — order matters. `ZeroDivisionError` is a subclass of `Exception`
(7.4 covers the full hierarchy), so a broad `except Exception` listed *first* matches
before a narrower one below it ever gets a chance — the specific block becomes dead
code:

In [ ]:
try:
    1 / 0
except Exception:
    print("Caught by broad Exception")          # this one wins
except ZeroDivisionError:
    print("Caught by specific ZeroDivisionError")   # unreachable — never runs

Several exception types can also share **one** `except` block, by grouping them in a
`tuple` (5.5):

In [ ]:
def parse_and_divide(a, b):
    try:
        return int(a) / int(b)
    except (ValueError, ZeroDivisionError) as e:   # either type lands here
        print(f"{type(e).__name__}: {e}")

parse_and_divide("x", 2)     # ValueError: invalid literal for int() with base 10: 'x'
parse_and_divide(10, 0)        # ZeroDivisionError: division by zero
print(parse_and_divide(10, 2))   # 5.0

**`try`/`except`/`else`** — 4.4 already introduced this: `else` runs only if the `try`
block completed with **no** exception raised at all. Another example, this time looking
something up in a `dict` (5.4):

In [ ]:
def get_score(scores, name):
    try:
        value = scores[name]
    except KeyError:
        print(f"{name} not found")
    else:
        print(f"{name} scored {value}")   # only runs if the lookup above didn't raise

get_score({"alice": 90}, "alice")   # alice scored 90
get_score({"alice": 90}, "bob")       # bob not found

**`finally`** — also introduced in 4.4: it runs **no matter what**. That includes when
the `try` or `except` block contains a `return` — `finally` still runs *before* the
function actually hands control back to the caller:

In [ ]:
def f():
    try:
        return 1
    finally:
        print("cleanup runs even though we're already returning")

print(f())   # cleanup runs even though we're already returning / 1

def g():
    try:
        raise ValueError("boom")
    except ValueError:
        return "handled"
    finally:
        print("g's cleanup always runs too")

print(g())   # g's cleanup always runs too / handled

**The structural rules** — not every combination of these four keywords is legal. The
valid order is always `try` → `except` (one or more) → `else` → `finally`, and:

| Rule | Statement |
|---|---|
| 1 | `try` must be followed by `except`, `finally`, or both |
| 2 | a bare `try` with none of `except`/`else`/`finally` is a `SyntaxError` |
| 3 | `else` is optional, but `try`/`else` with **no** `except` at all is invalid |
| 4 | `try` can be followed by **multiple** `except` blocks |
| 5 | `else` without at least one `except` is invalid (same rule as 3, from the other direction) |
| 6 | `try`/`except`/`else`/`finally` can be **nested** inside any of their own blocks |

`try` itself is always mandatory; beyond that, what's actually required is **at least
one `except`, or a `finally`, or both** — `else` is never enough on its own. Confirmed
directly against the parser:

In [ ]:
# compile() just parses the code without running it — enough to prove/disprove validity
# without actually executing a SyntaxError-laden statement at the top level of a cell
try:
    compile("try:\n    pass\n", "<test>", "exec")   # try with NOTHING after it
except SyntaxError as e:
    print("try alone -> SyntaxError:", e)

try:
    compile("try:\n    pass\nelse:\n    pass\n", "<test>", "exec")   # try/else, no except
except SyntaxError as e:
    print("try + else (no except) -> SyntaxError:", e)

`finally` on its own — with no `except` at all — is perfectly valid, since rule 1 only
needs `except` **or** `finally`. Likewise, `except`/`else` with no `finally` is fine too:

In [ ]:
compile("try:\n    pass\nfinally:\n    pass\n", "<test>", "exec")
print("try + finally, no except at all -> compiles fine")

compile("try:\n    pass\nexcept Exception:\n    pass\nelse:\n    pass\n", "<test>", "exec")
print("try + except + else, no finally -> compiles fine")

**Nesting** — a full `try`/`except`/`else`/`finally` can be placed inside any block of
an outer one (its `try`, an `except`, the `else`, or the `finally`). Each level handles
whatever it can; anything it doesn't catch keeps propagating out to the level above:

In [ ]:
def outer_op(a, b, c):
    try:
        result = a / b
        try:                                   # nested try, inside the outer try
            result = result / c
        except ZeroDivisionError:
            print("inner: cannot divide by c=0")
    except ZeroDivisionError:
        print("outer: cannot divide by b=0")
    else:
        print("no exceptions at all, result =", result)
    finally:
        print("outer_op finished")

outer_op(10, 2, 0)   # inner: cannot divide by c=0 / outer_op finished
outer_op(10, 0, 5)   # outer: cannot divide by b=0 / outer_op finished
outer_op(10, 2, 5)   # no exceptions at all, result = 1.0 / outer_op finished

### 7.3 Raising and Re-raising

Everything so far reacted to exceptions the interpreter raised on its own (division by
zero, a bad index, ...). `raise` lets *your own* code signal a problem the same way —
useful the moment a function detects a rule violation that isn't already a built-in
error (7.5 covers wrapping this in a custom exception type):

In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError(f"age cannot be negative: {age}")   # explicitly signal the problem
    return age

try:
    set_age(-1)
except ValueError as e:
    print("caught:", e)   # caught: age cannot be negative: -1

**Re-raising** — sometimes an `except` block wants to react to an exception (log it,
clean something up) without actually handling it, and let it keep propagating to
whoever called this code. A bare `raise` — with no exception after it — inside an
`except` block does exactly that: it re-raises the exception currently being handled,
unchanged:

In [ ]:
def process(x):
    try:
        return 10 / x
    except ZeroDivisionError:
        print("logging: division by zero happened")   # react, but don't swallow it
        raise   # bare raise -> re-raises the SAME exception, unchanged

try:
    process(0)
except ZeroDivisionError as e:
    print("caller caught it too:", e)
    # logging: division by zero happened
    # caller caught it too: division by zero

**Implicit chaining** — if a *new*, different exception is raised while an `except`
block is still handling one, Python doesn't discard the original. It attaches it to the
new exception's `__context__`, and an uncaught traceback shows both, joined by "During
handling of the above exception, another exception occurred":

In [ ]:
import traceback

try:
    try:
        1 / 0
    except ZeroDivisionError:
        int("x")   # a different, unrelated exception raised while handling the first
except ValueError as e:
    print(e.__context__)   # division by zero -> the original exception, kept automatically

    # reconstructing what an uncaught traceback would print:
    print("".join(traceback.format_exception_only(type(e.__context__), e.__context__)).strip())
    print("During handling of the above exception, another exception occurred:")
    print("".join(traceback.format_exception_only(type(e), e)).strip())
    # ZeroDivisionError: division by zero
    # During handling of the above exception, another exception occurred:
    # ValueError: invalid literal for int() with base 10: 'x'

**Explicit chaining — `raise ... from ...`** — this is the deliberate version of the
same idea: wrapping a low-level exception in a more meaningful custom one (7.5), while
recording *why*. It sets `__cause__` instead of `__context__`, and the traceback wording
changes to "The above exception was the direct cause of the following exception":

In [ ]:
class ConfigError(Exception):
    pass

def load_config(raw_value):
    try:
        return int(raw_value)
    except ValueError as e:
        raise ConfigError("config value must be an integer") from e   # explicit cause

try:
    load_config("not-a-number")
except ConfigError as e:
    print("caught:", e)                      # caught: config value must be an integer
    print("caused by:", repr(e.__cause__))   # caused by: ValueError("invalid literal for int() with base 10: 'not-a-number'")

**Suppressing chaining — `raise ... from None`** — when the original exception is
internal noise a caller shouldn't need to see (an implementation detail, not useful
context), `from None` drops it entirely instead of just hiding it:

In [ ]:
def load_config_quiet(raw_value):
    try:
        return int(raw_value)
    except ValueError:
        raise ConfigError("config value must be an integer") from None   # suppress the cause

try:
    load_config_quiet("not-a-number")
except ConfigError as e:
    print("caught:", e)                          # caught: config value must be an integer
    print("cause:", e.__cause__)                   # cause: None
    print("suppressed:", e.__suppress_context__)   # suppressed: True -> traceback won't show the ValueError either

### 7.4 The Exception Hierarchy

Every built-in exception is a **class**, and every one of them inherits from
`BaseException`. Almost all of them actually inherit from its subclass `Exception` —
the few direct `BaseException` subclasses (`SystemExit`, `KeyboardInterrupt`,
`GeneratorExit`) are meant to signal the program shutting down, not everyday errors, so
they're deliberately kept *outside* `Exception` — catching `Exception` never
accidentally swallows a `Ctrl+C`.

A simplified view of the tree, showing the exceptions used earlier in this chapter:

```text
BaseException
 ├── SystemExit, KeyboardInterrupt, GeneratorExit   (not errors — left alone)
 └── Exception
      ├── ArithmeticError
      │    └── ZeroDivisionError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── TypeError
      ├── ValueError
      ├── AttributeError
      ├── NameError
      ├── OSError
      │    └── FileNotFoundError
      └── RuntimeError
           └── RecursionError
```

In [ ]:
print(issubclass(ZeroDivisionError, ArithmeticError))   # True
print(issubclass(ArithmeticError, Exception))              # True
print(issubclass(Exception, BaseException))                 # True
print(issubclass(KeyboardInterrupt, Exception))              # False -> deliberately outside Exception

Because `except` matches by class **or any of its subclasses**, catching a base class
like `ArithmeticError` also catches every specific error under it — including ones
raised by code you haven't seen:

In [ ]:
try:
    1 / 0   # raises ZeroDivisionError specifically
except ArithmeticError as e:   # catches it anyway, since it's a subclass
    print("caught via ArithmeticError:", type(e).__name__)   # caught via ArithmeticError: ZeroDivisionError

**Common built-in exceptions** — a quick reference for the ones you'll hit constantly:

| Exception | Raised when |
|---|---|
| `ZeroDivisionError` | dividing by zero |
| `ValueError` | right type, but an invalid value (e.g. `int("x")`) |
| `IndexError` | sequence index out of range |
| `KeyError` | dict lookup with a missing key |
| `AttributeError` | accessing an attribute/method that doesn't exist on the object |
| `TypeError` | operation applied to an object of the wrong type |

In [ ]:
try:
    [1, 2, 3][5]
except IndexError as e:
    print("IndexError:", e)   # IndexError: list index out of range

try:
    {"a": 1}["b"]
except KeyError as e:
    print("KeyError:", e)   # KeyError: 'b'

try:
    None.upper()
except AttributeError as e:
    print("AttributeError:", e)   # AttributeError: 'NoneType' object has no attribute 'upper'

try:
    "2" + 2
except TypeError as e:
    print("TypeError:", e)   # TypeError: can only concatenate str (not "int") to str

**Common mistake** — a bare `except:` (no exception type at all) matches
`BaseException`, not `Exception`. That means it also swallows `SystemExit` and
`KeyboardInterrupt` — so `sys.exit()` silently fails to exit, and `Ctrl+C` can't stop the
program either. Always write `except Exception:` (or a specific type) instead of a bare
`except:`:

In [ ]:
try:
    raise SystemExit   # normally this would end the program
except:                  # bare except -> catches it anyway
    print("bare except caught SystemExit too")   # bare except caught SystemExit too

### 7.5 Custom Exceptions

Built-in exceptions only go so far — `ValueError` doesn't tell a caller *which* rule was
violated, and catching one from your own code can't be told apart from one raised by a
library doing something unrelated. A custom exception fixes both: it names the failure
precisely, and callers can catch it specifically without touching everything else that
happens to also raise `ValueError`.

Defining one is just subclassing `Exception` — no body needed at all:

In [ ]:
class NegativeAgeError(Exception):
    pass   # the name alone is the whole point — no extra behavior needed

def set_age(age):
    if age < 0:
        raise NegativeAgeError(f"age cannot be negative: {age}")
    return age

try:
    set_age(-5)
except NegativeAgeError as e:
    print("NegativeAgeError:", e)   # NegativeAgeError: age cannot be negative: -5

A custom exception can also override `__init__` to carry structured data along with it —
not just a message string, but attributes the `except` block can read back out:

In [ ]:
class InsufficientFundsError(Exception):
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"cannot withdraw {amount}, only {balance} available")

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount

try:
    withdraw(100, 150)
except InsufficientFundsError as e:
    print(e)                                # cannot withdraw 150, only 100 available
    print("short by", e.amount - e.balance)   # short by 50 -> extra attributes, not just text

**Custom exception hierarchies** — just as the built-ins form a tree (7.4), your own
exceptions can too. Define one base class for "anything that can go wrong in this app,"
then specific subclasses under it — callers can catch broadly via the base, or narrowly
via a specific subclass, exactly like with built-in exceptions:

In [ ]:
class AppError(Exception):
    pass   # base class for every error this app raises on purpose

class ValidationError(AppError):
    pass

class NotFoundError(AppError):
    pass

def lookup(record_id, valid):
    if not valid:
        raise ValidationError(f"invalid id: {record_id}")
    raise NotFoundError(f"no record for id: {record_id}")

for record_id, valid in [(-1, False), (42, True)]:
    try:
        lookup(record_id, valid)
    except AppError as e:   # catches ValidationError and NotFoundError both
        print(f"{type(e).__name__}: {e}")
        # ValidationError: invalid id: -1
        # NotFoundError: no record for id: 42

### 7.6 Exception Handling Practices

**Catch specific exceptions, not everything blindly.** A bare `except:` (7.4) or an
overly broad `except Exception:` can hide bugs that have nothing to do with what you
were actually guarding against. Catch only the exception type(s) the risky line can
actually raise.

**Never silently swallow an exception.** An empty `except: pass` makes a failure vanish
without a trace — the program looks fine, but something inside it quietly broke:

In [ ]:
def risky():
    raise ValueError("bad input")

try:
    risky()
except Exception:
    pass   # bug disappears without a trace — at minimum, log or print it

print("program continues, nobody knows risky() failed")   # program continues, nobody knows risky() failed

**Keep the `try` block as small as possible.** Wrap only the line(s) that can actually
raise, so an unrelated bug elsewhere in the same block doesn't get mistakenly caught and
misreported as "bad data":

In [ ]:
data = {"count": "5"}
try:
    n = int(data["count"])   # only the conversion can actually fail here
except (KeyError, ValueError) as e:
    print("bad data:", e)

print(n * 2)   # 10 -> the rest of the logic stays outside try, so it's obviously not the risky part

**EAFP vs. LBYL.** Python idiom generally favors **EAFP** ("Easier to Ask Forgiveness
than Permission" — try it, handle the failure) over **LBYL** ("Look Before You Leap" —
check first, then act). LBYL can also race: the condition can become false *between* the
check and the action; EAFP has no such gap because the check and the action are the same
step:

In [ ]:
d = {"a": 1}

# LBYL - check membership first, then access
if "b" in d:
    print(d["b"])
else:
    print("b missing (LBYL)")   # b missing (LBYL)

# EAFP - just try the access, handle the failure
try:
    print(d["b"])
except KeyError:
    print("b missing (EAFP)")   # b missing (EAFP)

**Always release resources, even on failure.** If a function opens something (a file, a
lock, a connection) and then raises partway through, that resource still has to be
cleaned up — `finally` (7.2) guarantees the cleanup line runs whether the body succeeded
or not:

In [ ]:
def process_file(path):
    f = open(path)
    try:
        raise ValueError("something went wrong while using the file")
    finally:
        f.close()
        print("file closed no matter what")   # file closed no matter what

try:
    process_file("/etc/hostname")
except ValueError as e:
    print("caught:", e)   # caught: something went wrong while using the file

### 7.9 Additional Exception Handling Concepts From Notes

Overflow bucket for this chapter — small or unclassified items that clearly belong to
this domain but not yet to a specific section above.

**The `assert` statement** — a sanity check, not general error handling: `assert
condition, message` raises `AssertionError(message)` if `condition` is falsy. It's meant
for catching *programmer* mistakes (an invariant that should never be false) during
development, not for validating user input — asserts can be stripped out entirely when
Python runs with the `-O` optimization flag, so input validation must never rely on
them:

In [ ]:
def set_temperature(celsius):
    assert celsius >= -273.15, f"temperature below absolute zero: {celsius}"
    return celsius

print(set_temperature(20))   # 20

try:
    set_temperature(-300)
except AssertionError as e:
    print("AssertionError:", e)   # AssertionError: temperature below absolute zero: -300

**Inside the exception object** — every exception stores its constructor arguments in
`.args`. `str(e)` renders it for humans (just the message), while `repr(e)` renders it
as Python-like code, including the class name — useful when logging, since it says
*which* exception type it was, not just its message:

In [ ]:
try:
    raise ValueError("bad value", 42)
except ValueError as e:
    print(e.args)   # ('bad value', 42)
    print(str(e))     # ('bad value', 42) -> str() of a multi-arg exception is its args tuple
    print(repr(e))   # ValueError('bad value', 42) -> includes the class name

In [ ]:
# --- 7. Exception Handling — scratch cell ---
# Experiments for this chapter. Promote anything worth keeping into the
# relevant section as a proper example cell.
